# Cat vs Dog Image Classifier (PyTorch)

Train a convolutional neural network to differentiate between cats and dogs using transfer learning with **MobileNetV2** in PyTorch.

**Dataset:** Oxford-IIIT Pet dataset (via `torchvision.datasets.OxfordIIITPet`), grouped into two classes: cat vs dog based on species label.

**Approach:**
1. Load and preprocess the dataset
2. Build a model using a pretrained MobileNetV2 as a feature extractor
3. Train the classification head
4. Fine-tune the top layers of the base model
5. Evaluate and run inference on sample images

## 1. Setup

In [ ]:
# Install dependencies if needed
# !pip install torch torchvision matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## 2. Load the Dataset

The Oxford-IIIT Pet dataset has 37 breeds; species labels (`0 = cat`, `1 = dog`) collapse it to a binary classification problem.

In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 32
DATA_ROOT = './data'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_full = datasets.OxfordIIITPet(
    root=DATA_ROOT, split='trainval', target_types='binary-category',
    download=True, transform=train_transform,
)
test_set = datasets.OxfordIIITPet(
    root=DATA_ROOT, split='test', target_types='binary-category',
    download=True, transform=eval_transform,
)

# 90/10 split of trainval into train/validation
val_size = int(0.1 * len(train_full))
train_size = len(train_full) - val_size
train_set, val_set = random_split(
    train_full, [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)
# Use eval transform (no augmentation) for the validation subset
val_set.dataset = datasets.OxfordIIITPet(
    root=DATA_ROOT, split='trainval', target_types='binary-category',
    download=False, transform=eval_transform,
)

CLASS_NAMES = ['cat', 'dog']
print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

In [ ]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# Preview a few examples
def denormalize(img):
    img = img.cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

images, labels = next(iter(train_loader))
plt.figure(figsize=(10, 6))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(denormalize(images[i]))
    plt.title(CLASS_NAMES[int(labels[i])])
    plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Build the Model

Use MobileNetV2 pretrained on ImageNet as a frozen feature extractor with a small classification head on top.

In [ ]:
weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
model = models.mobilenet_v2(weights=weights)

# Freeze the base
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier with a binary head (1 logit)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(in_features, 1),
)
model = model.to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,} | Trainable: {trainable:,}")

## 4. Training Utilities

In [ ]:
criterion = nn.BCEWithLogitsLoss()

def run_epoch(loader, train=False, optimizer=None):
    model.train(train)
    total_loss = 0.0
    correct = 0
    n = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == labels).sum().item()
        n += images.size(0)
    return total_loss / n, correct / n

history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

def train_for(epochs, optimizer, label='phase'):
    for epoch in range(epochs):
        tr_loss, tr_acc = run_epoch(train_loader, train=True, optimizer=optimizer)
        val_loss, val_acc = run_epoch(val_loader, train=False)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(val_acc)
        print(f"[{label}] epoch {epoch+1}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")

## 5. Train the Classification Head

In [ ]:
INITIAL_EPOCHS = 5
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
train_for(INITIAL_EPOCHS, optimizer, label='head')

## 6. Fine-Tune the Top Layers

Unfreeze the later blocks of MobileNetV2's `features` and continue training with a much smaller learning rate.

In [ ]:
# Unfreeze from block 14 onward (MobileNetV2 has 19 feature blocks: 0..18)
FINE_TUNE_AT = 14
for i, block in enumerate(model.features):
    for param in block.parameters():
        param.requires_grad = i >= FINE_TUNE_AT

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params after unfreezing: {trainable:,}")

FINE_TUNE_EPOCHS = 5
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
train_for(FINE_TUNE_EPOCHS, optimizer, label='fine-tune')

## 7. Evaluate on the Test Set

In [ ]:
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history['train_acc'], label='Training accuracy')
plt.plot(history['val_acc'], label='Validation accuracy')
plt.axvline(INITIAL_EPOCHS - 1, color='gray', linestyle='--', label='Start fine-tuning')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training history')
plt.show()

## 8. Run Inference on Sample Test Images

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    logits = model(images.to(device))
    probs = torch.sigmoid(logits).cpu().numpy().flatten()
preds = (probs > 0.5).astype(int)

plt.figure(figsize=(12, 8))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(denormalize(images[i]))
    actual = CLASS_NAMES[int(labels[i])]
    predicted = CLASS_NAMES[int(preds[i])]
    color = 'green' if preds[i] == int(labels[i]) else 'red'
    plt.title(f"pred: {predicted} ({probs[i]:.2f})\nactual: {actual}", color=color)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 9. Save the Model

In [ ]:
torch.save(model.state_dict(), 'cat_vs_dog_classifier.pt')
print('Model weights saved to cat_vs_dog_classifier.pt')